# MPC-DAgger so với MPC-Inject trên GPU (Colab)

So sánh hai cách dùng MPC để giúp PPO, cùng thầy **MPOPI** (K = 8 × 4 vòng, H = 20),
không nhiễu, trên `Mjlab-Cartpole-Swingup` (64 env PPO, 200 vòng, đánh giá mỗi 10 vòng):

| Phương pháp | Cách dùng MPC |
|---|---|
| **PPO** | Không dùng MPC (mốc so sánh) |
| **MPC-DAgger** | 8 env phụ do policy tự chạy; MPOPI gắn nhãn trạng thái policy đi qua; loss bắt chước (BC) giảm từ 1 về 0 sau 30 vòng; thu nhãn trong 20 vòng đầu |
| **MPC-Inject** | 8 env phụ do MPOPI tự chạy trong 20 vòng đầu (2560 mẫu, giữ cố định); **mọi** batch PPO có 25% là mẫu MPC, coi như dữ liệu của chính PPO (không hiệu chỉnh IS, không BC), suốt 200 vòng. Theo ý tưởng bài MPC-Injection (arXiv 2606.26392), chuyển sang PPO |

**Giả thuyết đặt trước** (10 seed, Wilcoxon cặp hai phía, α = 0,05):
- **H1:** AUC của MPC-Inject > PPO.
- **H2:** AUC của MPC-Inject so với MPC-DAgger (hai phía, không đoán trước hướng).

**Tiết kiệm GPU:** mặc định dùng lại seed 500–509 và thư mục Drive `mpc_dagger_gpu` của
lần xác nhận MPC-DAgger trước. PPO và MPC-DAgger đã có kết quả ở đó sẽ tự được bỏ qua,
nên chỉ phải chạy 10 lần MPC-Inject. Nếu thư mục đó không còn, notebook chạy cả 3.

**Trước khi chạy:** *Runtime → Change runtime type → GPU* (T4 là đủ).

## 0. Cấu hình

In [ ]:
#@title Cấu hình chung
REPO_URL = "https://github.com/TamasTran/mjlab_MPOPI.git"  #@param {type:"string"}
BRANCH = "mpc-stage1"  #@param {type:"string"}
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
#@markdown Dùng lại kết quả PPO và MPC-DAgger của lần chạy trước (cùng thư mục Drive):
REUSE_PREVIOUS = True  #@param {type:"boolean"}
FIRST_SEED = 500  #@param {type:"integer"}
NUM_SEEDS = 10  #@param {type:"integer"}
ITERATIONS = 200  #@param {type:"integer"}
INJECT_FRACTION = 0.25  #@param {type:"number"}
PARALLEL = 2  #@param {type:"integer"}

import os

REPO_DIR = "/content/mjlab_MPOPI"
if SAVE_TO_DRIVE:
  from google.colab import drive

  drive.mount("/content/drive")
  base = "/content/drive/MyDrive"
else:
  base = "/content"
OUT_ROOT = f"{base}/mpc_dagger_gpu" if REUSE_PREVIOUS else f"{base}/mpc_dagger_vs_inject_gpu"
os.makedirs(OUT_ROOT, exist_ok=True)
SEEDS = list(range(FIRST_SEED, FIRST_SEED + NUM_SEEDS))
print("Kết quả lưu tại:", OUT_ROOT, "| seed:", SEEDS)

In [ ]:
#@title Hàm tiện ích: chạy lệnh và in log trực tiếp
import subprocess


def run(cmd: str, cwd: str | None = None) -> None:
  """Run a shell command, stream its output, and raise if it fails."""
  print(f"$ {cmd}", flush=True)
  proc = subprocess.Popen(
    cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env={**os.environ},
  )
  assert proc.stdout is not None
  for line in proc.stdout:
    print(line, end="", flush=True)
  if proc.wait() != 0:
    raise RuntimeError(f"Command failed with exit code {proc.returncode}: {cmd}")

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi

## 2. Lấy code

Nhánh `mpc-stage1` phải được push bản mới nhất (có commit
`Add MPC-Injection for PPO ...`).

In [ ]:
import shutil

if os.path.exists(REPO_DIR):
  shutil.rmtree(REPO_DIR)
run(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}")
run("git log --oneline -1", cwd=REPO_DIR)
config_src = open(f"{REPO_DIR}/src/mjlab/rl/mpopi/config.py").read()
if "inject_fraction" not in config_src or "bc_floor" not in config_src:
  raise RuntimeError(
    "Code trên GitHub đã cũ (thiếu inject_fraction). "
    "Hãy push bản mới nhất của nhánh mpc-stage1 rồi chạy lại ô này."
  )
print("OK: code có MPC-DAgger và MPC-Inject.")

## 3. Cài đặt môi trường

`uv sync` **không** kèm `--extra cpu` để có PyTorch bản CUDA. Lần đầu mất khoảng 5–10 phút.

In [ ]:
run("curl -LsSf https://astral.sh/uv/install.sh | sh")
os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"
run("uv sync", cwd=REPO_DIR)
run(
  "uv run python -c \"import torch; print('torch', torch.__version__, 'cuda', "
  "torch.cuda.is_available(), torch.cuda.get_device_name(0))\"",
  cwd=REPO_DIR,
)

## 4. Test trên GPU

Nếu ô này lỗi thì **dừng lại**, đừng chạy các mục sau.

In [ ]:
run("uv run pytest tests/test_mpc_ppo.py tests/test_mpc_sampling.py -q -p no:warnings", cwd=REPO_DIR)

## 5. Các phương pháp và smoke test

Chạy thử MPC-Inject 5 vòng để kiểm tra đường chạy đầy đủ và ước lượng thời gian.

In [ ]:
COMMON = (
  "uv run python scripts/benchmarks/mpopi_benchmark.py --task Mjlab-Cartpole-Swingup "
  "--device cuda:0 --num-envs 64 --eval-every 10 --eval-steps 400 --seeds 1 "
  "--progress-every 20"
)
MPOPI_TEACHER = "--mpc.planner.num-samples 8 --mpc.planner.iterations 4 --mpc.execution-std 0"
ARMS = {
  # Folder names match the previous DAgger notebook so its results are reused.
  "A_ppo": "--arms A_ppo",
  "dagger_mpopi": f"--arms D_mpc_bc_only --mpc.driver policy {MPOPI_TEACHER}",
  "inject_mpopi": (
    f"--arms D_mpc_naive --mpc.bc-coef 0 --mpc.max-age None --mpc.buffer-segments 20 "
    f"--mpc.inject-fraction {INJECT_FRACTION} {MPOPI_TEACHER}"
  ),
}
NAMES = {"A_ppo": "PPO", "dagger_mpopi": "MPC-DAgger", "inject_mpopi": "MPC-Inject"}

import time

start = time.time()
run(
  f"{COMMON} --iterations 5 --seed-offset 0 {ARMS['inject_mpopi']} "
  f"--mpc.collect-iterations 3 --out-dir {OUT_ROOT}/smoke_inject",
  cwd=REPO_DIR,
)
print(f"Smoke test xong sau {time.time() - start:.0f} s")

## 6. Chạy thí nghiệm chính

Lần chạy nào đã có `curves.csv` thì bỏ qua, nên có thể chạy lại ô này sau khi Colab ngắt.
Log từng lần chạy: `OUT_ROOT/logs/<phương pháp>_s<seed>.log`.

In [ ]:
import concurrent.futures as cf
import glob

LOG_DIR = f"{OUT_ROOT}/logs"
os.makedirs(LOG_DIR, exist_ok=True)


def job(arm: str, seed: int) -> str:
  out = f"{OUT_ROOT}/{arm}/s{seed}"
  if os.path.exists(f"{out}/curves.csv"):
    return f"{NAMES[arm]} seed {seed}: đã có, bỏ qua"
  cmd = f"{COMMON} --iterations {ITERATIONS} --seed-offset {seed} {ARMS[arm]} --out-dir {out}"
  env = {**os.environ, "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"}
  t0 = time.time()
  with open(f"{LOG_DIR}/{arm}_s{seed}.log", "w") as log:
    rc = subprocess.call(cmd, shell=True, cwd=REPO_DIR, stdout=log, stderr=subprocess.STDOUT, env=env)
  status = "xong" if rc == 0 else f"LỖI (exit {rc}), xem log"
  return f"{NAMES[arm]} seed {seed}: {status} sau {time.time() - t0:.0f} s"


jobs = [(arm, seed) for seed in SEEDS for arm in ARMS]
with cf.ThreadPoolExecutor(max_workers=PARALLEL) as pool:
  for msg in pool.map(lambda a: job(*a), jobs):
    print(msg, flush=True)

In [ ]:
#@title Xem tiến độ (chạy trong lúc mục 6 đang chạy)
for path in sorted(glob.glob(f"{OUT_ROOT}/logs/*.log")):
  lines = [l for l in open(path) if l.startswith("[")]
  print(os.path.basename(path), "|", lines[-1].strip()[:110] if lines else "đang khởi động / thu MPC")

## 7. Phân tích

AUC = điểm trung bình suốt quá trình train (chỉ số chính). So sánh cặp theo seed.
Thời gian đạt 0,040 được ước lượng từ thời gian thu MPC thực đo cộng thời gian PPO
trung bình mỗi vòng.

In [ ]:
import csv
import math

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

THRESH, MAX = 0.040, 0.05
COLORS = {"A_ppo": "#555555", "dagger_mpopi": "#d62728", "inject_mpopi": "#1f77b4"}


def load(arm):
  runs = {}
  for p in glob.glob(f"{OUT_ROOT}/{arm}/s*/curves.csv"):
    for r in csv.DictReader(open(p)):
      if int(r["seed"]) in SEEDS:
        runs.setdefault(int(r["seed"]), []).append(r)
  return runs


def fl(x):
  try:
    return float(x)
  except ValueError:
    return math.nan


def run_stats(rows):
  ev = [(int(r["iteration"]), fl(r["eval_return"])) for r in rows]
  ev = [(i, s) for i, s in ev if not math.isnan(s)]
  scores = [s for _, s in ev]
  tail = max(1, len(scores) // 10)
  collect = [fl(r.get("mpc/collect_seconds", "nan")) for r in rows]
  collect = [0.0 if math.isnan(c) else c for c in collect]
  total = fl(rows[0]["run_seconds"])
  ppo_per_it = (total - sum(collect)) / len(rows)
  hit = next((i for i, s in ev if s >= THRESH), None)
  hit_s = None if hit is None else sum(collect[: hit + 1]) + ppo_per_it * (hit + 1)
  return dict(auc=sum(scores) / len(scores), final=sum(scores[-tail:]) / tail,
              hit=hit, hit_s=hit_s, seconds=total, mpc_s=sum(collect), curve=dict(ev))


table = {arm: {s: run_stats(r) for s, r in load(arm).items()} for arm in ARMS}
print(f"{'phương pháp':>12} {'n':>3} {'AUC':>7} {'cuối':>7} {'đạt 0.040':>10} {'giây đạt (trung vị)':>20} {'giây/lần':>9}")
for arm, t in table.items():
  st = [t[s] for s in sorted(t)]
  if not st:
    print(f"{NAMES[arm]:>12}: chưa có kết quả")
    continue
  hits = [x["hit_s"] for x in st if x["hit_s"] is not None]
  print(f"{NAMES[arm]:>12} {len(st):>3} {np.mean([x['auc'] for x in st]):7.4f} "
        f"{np.mean([x['final'] for x in st]):7.4f} {len(hits):>6}/{len(st):<3} "
        f"{(np.median(hits) if hits else float('nan')):>20.0f} {np.mean([x['seconds'] for x in st]):9.0f}")


def paired(a, b, label):
  ta, tb = table[a], table[b]
  seeds = sorted(set(ta) & set(tb))
  if len(seeds) < 2:
    print(f"{label}: chưa đủ dữ liệu")
    return None
  for m in ("auc", "final"):
    d = [ta[s][m] - tb[s][m] for s in seeds]
    p = stats.wilcoxon(d).pvalue if any(d) else math.nan
    print(f"{label} [{m}]: {np.mean(d):+.4f}, {sum(x > 0 for x in d)}/{len(d)} seed dương, p = {p:.3g}")
  d = [ta[s]["auc"] - tb[s]["auc"] for s in seeds]
  return np.mean(d), stats.wilcoxon(d).pvalue if any(d) else math.nan


print()
h1 = paired("inject_mpopi", "A_ppo", "H1  MPC-Inject − PPO       ")
h2 = paired("inject_mpopi", "dagger_mpopi", "H2  MPC-Inject − MPC-DAgger")
paired("dagger_mpopi", "A_ppo", "    MPC-DAgger − PPO       ")
if h1:
  print(f"\nH1 (AUC MPC-Inject > PPO, p < 0.05): {'XÁC NHẬN' if h1[0] > 0 and h1[1] < 0.05 else 'CHƯA XÁC NHẬN'}")
if h2:
  verdict = "khác biệt có ý nghĩa" if h2[1] < 0.05 else "chưa có khác biệt có ý nghĩa"
  print(f"H2 (MPC-Inject so với MPC-DAgger, p < 0.05): {verdict} (chênh {h2[0]:+.4f})")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [1.5, 1]})
ax = axes[0]
for arm, t in table.items():
  if not t:
    continue
  its = sorted(set.intersection(*(set(x["curve"]) for x in t.values())))
  y = np.array([[t[s]["curve"][i] / MAX for i in its] for s in t])
  ax.plot(its, y.mean(0), color=COLORS[arm], lw=2, label=f"{NAMES[arm]} (TB {len(t)} seed)")
  ax.fill_between(its, y.min(0), y.max(0), color=COLORS[arm], alpha=0.1, lw=0)
ax.axvspan(0, 20, color="#fde0c5", alpha=0.4, lw=0)
ax.set_xlabel("Vòng lặp PPO")
ax.set_ylabel("Điểm chuẩn hóa")
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)
ax.legend(loc="lower right")
ax.set_title("Đường học (vùng cam: 20 vòng thu dữ liệu MPC)")
ax = axes[1]
for i, arm in enumerate(ARMS):
  t = table[arm]
  for s in sorted(t):
    ax.plot(i + np.random.default_rng(s).uniform(-0.12, 0.12), t[s]["auc"] / MAX, "o", color=COLORS[arm], ms=5)
  if t:
    ax.hlines(np.mean([x["auc"] for x in t.values()]) / MAX, i - 0.3, i + 0.3, color="black", lw=2)
ax.set_xticks(range(len(ARMS)), [NAMES[a] for a in ARMS])
ax.set_ylabel("AUC chuẩn hóa")
ax.set_ylim(0, 1.0)
ax.grid(alpha=0.3)
ax.set_title("AUC từng seed (vạch đen = trung bình)")
plt.tight_layout()
plt.savefig(f"{OUT_ROOT}/dagger_vs_inject_gpu.png", dpi=150)
plt.show()

## 8. Tải kết quả về

In [ ]:
shutil.make_archive("/content/mpc_dagger_vs_inject_results", "zip", OUT_ROOT)
from google.colab import files

files.download("/content/mpc_dagger_vs_inject_results.zip")